# Conformer Ensemble — SMOKE TEST
Minimal-runtime copy of `conformer_ensemble.ipynb` for verifying the GitHub Actions pipeline works end-to-end. Uses a small molecule subsample, 2 conformers, and 3 training epochs.

In [ ]:
import os
import torch
print('Current directory:', os.getcwd())
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
import os, json, copy, random, pickle
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import SchNet

from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score
from sklearn.calibration import calibration_curve

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog('rdApp.*')

from tdc.single_pred import Tox

import warnings
warnings.filterwarnings('ignore')

# ---- SMOKE TEST KNOBS ----
SEED = 42
N_CONFORMERS = 2          # was 10/3
MAX_MOLS = 80             # subsample to keep wall time tiny
MAX_EPOCHS = 3            # was 120/25
PATIENCE = 2
BATCH_SIZE = 16
# --------------------------

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DATA_PATH = Path('data')
MODELS_DIR = Path('output/models/smoke'); MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = Path('output/results/smoke'); RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = Path('data/conformer_cache_smoke'); CACHE_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Load ClinTox (subsampled) + scaffold split

In [ ]:
def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    try: return MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    except Exception: return None

def scaffold_split(df, smiles_col='Drug', frac_train=0.7, frac_val=0.1, seed=42):
    scaffolds = defaultdict(list)
    for idx, row in df.iterrows():
        scaffolds[get_scaffold(row[smiles_col]) or ''].append(idx)
    rng = np.random.RandomState(seed)
    groups = sorted(scaffolds.values(), key=lambda g: (-len(g), rng.random()))
    n = len(df); n_train = int(frac_train * n); n_val = int(frac_val * n)
    tr, va, te = [], [], []
    for g in groups:
        if len(tr) + len(g) <= n_train: tr.extend(g)
        elif len(va) + len(g) <= n_val: va.extend(g)
        else: te.extend(g)
    return {'train': df.loc[tr].reset_index(drop=True),
            'valid': df.loc[va].reset_index(drop=True),
            'test':  df.loc[te].reset_index(drop=True)}

data = Tox(name='ClinTox', path=str(DATA_PATH))
df = data.get_data()
df = df.sample(n=min(MAX_MOLS, len(df)), random_state=SEED).reset_index(drop=True)
split = scaffold_split(df, seed=SEED)
for k, v in split.items():
    pos = int((v['Y'] == 1).sum())
    print(f'  {k}: n={len(v)}  positive={pos}')

## 2. Generate conformers (small)

In [ ]:
def smiles_to_conformers(smiles, n_conformers=2, seed=42):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    mol_h = Chem.AddHs(mol)
    params = AllChem.ETKDGv3(); params.randomSeed = seed
    ids = AllChem.EmbedMultipleConfs(mol_h, numConfs=n_conformers, params=params)
    if len(ids) == 0:
        ids = AllChem.EmbedMultipleConfs(mol_h, numConfs=n_conformers, randomSeed=seed, useRandomCoords=True)
    if len(ids) == 0: return None
    for cid in ids:
        try: AllChem.UFFOptimizeMolecule(mol_h, confId=int(cid), maxIters=50)
        except Exception: pass
    heavy_idx = [a.GetIdx() for a in mol_h.GetAtoms() if a.GetAtomicNum() > 1]
    z = np.array([mol_h.GetAtomWithIdx(i).GetAtomicNum() for i in heavy_idx], dtype=np.int64)
    positions = []
    for cid in ids:
        conf = mol_h.GetConformer(int(cid))
        pos = np.array([[conf.GetAtomPosition(i).x, conf.GetAtomPosition(i).y, conf.GetAtomPosition(i).z]
                       for i in heavy_idx], dtype=np.float32)
        positions.append(pos)
    return z, positions

def build_cache(df, cache_path, n_conformers=2, seed=42):
    if cache_path.exists():
        with open(cache_path, 'rb') as f: return pickle.load(f)
    cache = {}
    for i, row in df.iterrows():
        out = smiles_to_conformers(row['Drug'], n_conformers=n_conformers, seed=seed)
        if out is None: continue
        cache[i] = {'z': out[0], 'positions': out[1], 'y': float(row['Y']), 'smiles': row['Drug']}
    with open(cache_path, 'wb') as f: pickle.dump(cache, f)
    return cache

train_cache = build_cache(split['train'], CACHE_DIR / f'train_n{N_CONFORMERS}.pkl', N_CONFORMERS, SEED)
valid_cache = build_cache(split['valid'], CACHE_DIR / f'valid_n{N_CONFORMERS}.pkl', N_CONFORMERS, SEED)
test_cache  = build_cache(split['test'],  CACHE_DIR / f'test_n{N_CONFORMERS}.pkl',  N_CONFORMERS, SEED)
print(f'Train: {len(train_cache)}, Valid: {len(valid_cache)}, Test: {len(test_cache)}')

## 3. Build PyG dataset

In [ ]:
def conformer_to_data(z, pos, y, mol_id, conf_id):
    return Data(z=torch.tensor(z, dtype=torch.long),
                pos=torch.tensor(pos, dtype=torch.float),
                y=torch.tensor([y], dtype=torch.float),
                mol_id=int(mol_id), conf_id=int(conf_id))

def cache_to_list(cache, mode='single'):
    out = []
    for mol_id, item in cache.items():
        if mode == 'single':
            out.append(conformer_to_data(item['z'], item['positions'][0], item['y'], mol_id, 0))
        else:
            for cid, pos in enumerate(item['positions']):
                out.append(conformer_to_data(item['z'], pos, item['y'], mol_id, cid))
    return out

train_data = cache_to_list(train_cache, 'single')
valid_data = cache_to_list(valid_cache, 'single')
test_data_single = cache_to_list(test_cache, 'single')
test_data_all = cache_to_list(test_cache, 'all')
print(f'train={len(train_data)} valid={len(valid_data)} test_single={len(test_data_single)} test_all={len(test_data_all)}')

## 4. Tiny SchNet + 3-epoch training

In [ ]:
class SchNetClassifier(nn.Module):
    def __init__(self, hidden=32, num_filters=32, num_interactions=2, cutoff=10.0, num_gaussians=25):
        super().__init__()
        self.encoder = SchNet(hidden_channels=hidden, num_filters=num_filters,
                              num_interactions=num_interactions, num_gaussians=num_gaussians,
                              cutoff=cutoff)
    def forward(self, z, pos, batch):
        return self.encoder(z, pos, batch).squeeze(-1)

def train(train_data, valid_data, max_epochs=MAX_EPOCHS, patience=PATIENCE, batch_size=BATCH_SIZE, lr=5e-4):
    train_loader = PyGDataLoader(train_data, batch_size=batch_size, shuffle=True, drop_last=False)
    valid_loader = PyGDataLoader(valid_data, batch_size=batch_size)
    model = SchNetClassifier().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-6)
    ys = torch.cat([d.y for d in train_data]).view(-1)
    n_pos = max(int(ys.sum().item()), 1); n_neg = max(int(len(ys)) - n_pos, 1)
    pw = torch.tensor([n_neg / n_pos], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    best_val = float('inf'); best_state = copy.deepcopy(model.state_dict()); bad = 0
    for epoch in range(1, max_epochs + 1):
        model.train(); n = 0; running = 0.0
        for batch in train_loader:
            batch = batch.to(device); opt.zero_grad()
            out = model(batch.z, batch.pos, batch.batch)
            loss = criterion(out, batch.y.view(-1)); loss.backward(); opt.step()
            running += loss.item() * batch.num_graphs; n += batch.num_graphs
        train_loss = running / max(n, 1)
        model.eval(); n = 0; running = 0.0
        with torch.no_grad():
            for batch in valid_loader:
                batch = batch.to(device)
                out = model(batch.z, batch.pos, batch.batch)
                running += criterion(out, batch.y.view(-1)).item() * batch.num_graphs; n += batch.num_graphs
        val_loss = running / max(n, 1)
        print(f'  ep {epoch}  train={train_loss:.4f}  val={val_loss:.4f}')
        if val_loss + 1e-4 < best_val:
            best_val = val_loss; best_state = copy.deepcopy(model.state_dict()); bad = 0
        else:
            bad += 1
            if bad >= patience: print('  early stop'); break
    model.load_state_dict(best_state)
    return model

model = train(train_data, valid_data)
torch.save(model.state_dict(), MODELS_DIR / f'SchNet_smoke_seed{SEED}.pt')

## 5. Inference + per-molecule aggregation

In [ ]:
def predict_all(model, data_list, batch_size=32):
    loader = PyGDataLoader(data_list, batch_size=batch_size)
    rows = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            probs = torch.sigmoid(model(batch.z, batch.pos, batch.batch)).cpu().numpy()
            ys = batch.y.view(-1).cpu().numpy()
            mol_ids = batch.mol_id.cpu().numpy() if hasattr(batch.mol_id, 'cpu') else np.array(batch.mol_id)
            conf_ids = batch.conf_id.cpu().numpy() if hasattr(batch.conf_id, 'cpu') else np.array(batch.conf_id)
            for mid, cid, y, p in zip(mol_ids, conf_ids, ys, probs):
                rows.append({'mol_id': int(mid), 'conf_id': int(cid), 'y_true': float(y), 'prob': float(p)})
    return pd.DataFrame(rows)

single_df = predict_all(model, test_data_single)
all_df = predict_all(model, test_data_all)
per_mol = all_df.groupby('mol_id').agg(
    y_true=('y_true', 'first'),
    prob_mean=('prob', 'mean'),
    prob_std=('prob', 'std'),
    n_conformers=('conf_id', 'nunique'),
).reset_index()
per_mol.to_csv(RESULTS_DIR / 'per_molecule.csv', index=False)
print(per_mol.head())

## 6. Quick summary metrics + plot

In [ ]:
def cls_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        'auroc': float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else float('nan'),
        'f1':    float(f1_score(y_true, y_pred, zero_division=0)),
        'accuracy':  float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall':    float(recall_score(y_true, y_pred, zero_division=0)),
    }

summary_rows = [
    {'method': 'SchNet (1 conformer)',
     **cls_metrics(single_df['y_true'].values, single_df['prob'].values)},
    {'method': f'SchNet ({N_CONFORMERS}-conformer mean)',
     **cls_metrics(per_mol['y_true'].values, per_mol['prob_mean'].values)},
]
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULTS_DIR / 'summary.csv', index=False)
print(summary_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(per_mol['prob_std'].dropna(), bins=15, color='#4A90D9', edgecolor='black', alpha=0.85)
ax.set_xlabel('Std of predicted probability'); ax.set_ylabel('# molecules')
ax.set_title(f'Smoke test: per-molecule variance ({N_CONFORMERS} conformers)')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'variance_hist.png', dpi=120)
plt.show()

print('Smoke test complete.')